# Perturbation XAI cho hai checkpoint VG/PSG

Chạy trên Colab GPU sau `train_vg_psg_for_xai_colab.ipynb`. Notebook dùng **`duong/xai/model.py` + `EdgeContributionExplainer` + `XAIEvaluator` + `score_without_edges`**. Mỗi lần xóa cạnh, adapter dựng lại nhánh graph/QA của checkpoint thật trên đủ top-100 ứng viên, giữ α và CLIP pool cố định. Đối chứng ngẫu nhiên khớp số cạnh `contains`/`sg_rel`/`concept`. Đồ thị đưa vào XAI chỉ gồm các cạnh được sàng lọc ở vùng 2-hop; các cạnh ngoài vùng này vẫn còn trong graph của model.

Mỗi dataset chọn 12 caption test theo seed 0: 3 direct/indirect × đúng top-1/sai ngoài top-10, chỉ khi ảnh đúng ở top-100 CLIP. Kết quả là case study, không phải ước lượng toàn test. Giới hạn 40 cạnh quan hệ nhãn được xét mỗi case; báo coverage. Cạnh có provenance nhưng evidence ngữ nghĩa/thị giác vẫn là `uncertain`. Xóa cạnh có ảnh hưởng lên top-1 không đồng nghĩa cải thiện hạng của ảnh đúng.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, importlib.util, json, statistics, subprocess, sys
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'graphml-project3'
RUNS = REPO / 'duong/runs/qa_vg_psg_xai'
MANIFEST = None  # None = manifest huấn luyện mới nhất
SEEDS = (0, 1, 2)
N_PER_STRATUM = 3
MAX_REL_EDGES = 40
RANDOM_REPEATS = 20
EXPLANATION_SIZES = (1, 3, 5)
SELECTION_SEED = 42
assert (REPO / 'duong/xai/model.py').is_file(), 'Đồng bộ repo lên Drive.'
if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO / 'huy')], check=True)
sys.path[:0] = [str(REPO / 'duong'), str(REPO / 'huy/src')]
import torch
from xai.model import HuyQACaseModel, MODEL_SPECS
from xai.explain import EdgeContributionExplainer
from xai.evaluate_xai import XAIEvaluator
from xai.perturbation import rank_of, score_without_edges
from xai.io import explanation_to_dict
from mmxai.data.task1 import load_task1
from mmxai.graph.build import build_graph
from mmxai.graph.qa import QAData, QAScorer, qa_scores
from mmxai.experiments.t1_graph import load_eval_module
assert torch.cuda.is_available(), 'Chọn GPU trong Colab Runtime.'
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
if MANIFEST is None:
    manifests = sorted(RUNS.glob('dual-*_manifest.json'), key=lambda p: p.stat().st_mtime)
    assert manifests, f'Không tìm thấy manifest trong {RUNS}'
    MANIFEST = manifests[-1]
MANIFEST = Path(MANIFEST)
m = json.loads(MANIFEST.read_text())
assert len(m['runs']) == len(MODEL_SPECS) * len(SEEDS)
records = {(r['dataset'], int(r['seed'])): r for r in m['runs']}
assert set(records) == {(name, seed) for name in MODEL_SPECS for seed in SEEDS}
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
for rel, expected in m['source_sha256'].items():
    assert sha256(REPO / rel) == expected, f'Mã nguồn đổi từ khi huấn luyện: {rel}'
for name, spec in MODEL_SPECS.items():
    bundle_dir = DRIVE / spec.bundle_dir
    for rel, expected in m['bundle_sha256'][name].items():
        assert sha256(bundle_dir / rel) == expected, f'{name}: dữ liệu đổi: {rel}'
    for seed in SEEDS:
        rec = records[name, seed]
        checkpoint = Path(rec['run_dir']) / 'model.pt'
        assert checkpoint.is_file() and sha256(checkpoint) == rec['checkpoint_sha256']
OUT = RUNS / datetime.now(timezone.utc).strftime('xai-%Y%m%d-%H%M%S')
OUT.mkdir(parents=True, exist_ok=False)
print('Manifest:', MANIFEST, '| output:', OUT)


In [ ]:
all_rows, all_explanations, selected_cases = [], [], []
for name, spec in MODEL_SPECS.items():
    bundle_dir = DRIVE / spec.bundle_dir
    ev = load_eval_module(bundle_dir)
    protocol = ev.Protocol.load(bundle_dir)
    bundle = load_task1(bundle_dir, bundle_dir / 'huy_extra_cn')
    cap_concepts = {int(c['caption_id']): c['concepts'] for c in bundle.captions}
    cands = protocol.candidates('test', None)
    seed0 = records[name, 0]
    saved_seed0 = json.loads((Path(seed0['run_dir']) / 'ranks_test_full.json').read_text())['ranks']
    assert len(saved_seed0) == len(cands['q_cap'])
    rng = np.random.default_rng(SELECTION_SEED)
    case_indices = []
    for group in ('direct', 'indirect'):
        for outcome in ('success', 'failure'):
            pool = [j for j, rank in enumerate(saved_seed0)
                    if cands['group'][j] == group and int(cands['pos'][j]) >= 0
                    and ((int(rank) == 1) if outcome == 'success' else (int(rank) > 10))]
            assert len(pool) >= N_PER_STRATUM, f'{name}: thiếu case {group}/{outcome}: {len(pool)}'
            for j in sorted(rng.choice(pool, N_PER_STRATUM, replace=False).tolist()):
                case_indices.append((int(j), group, outcome))
                selected_cases.append({'dataset': name, 'candidate_row': int(j),
                                       'caption_id': int(cands['q_cap'][j]), 'group': group,
                                       'seed0_outcome': outcome,
                                       'ground_truth_image_id': int(cands['q_img'][j])})
    x_img = torch.from_numpy(bundle.image_emb).to(device)
    for seed in SEEDS:
        rec = records[name, seed]
        run_dir = Path(rec['run_dir'])
        cfg = json.loads((run_dir / 'config.json').read_text())
        graph = build_graph(bundle, variant=cfg['graph']['variant'],
                            min_rel_count=int(cfg['graph'].get('min_rel_count', 1)), seed=seed,
                            rel_merge=cfg['graph'].get('rel_merge', 'directed'))
        stats = json.loads((run_dir / 'metrics.json').read_text())['run']['extra']['graph']
        assert graph.stats['n_label'] == stats['n_label']
        assert graph.stats['E_concept'] == stats['E_concept']
        qd = QAData(graph, cap_concepts, device)
        model = QAScorer(bundle.image_emb.shape[1], **cfg['gnn']).to(device)
        model.load_state_dict(torch.load(run_dir / 'model.pt', map_location='cpu', weights_only=True))
        model.eval()
        saved_ranks = json.loads((run_dir / 'ranks_test_full.json').read_text())['ranks']
        assert len(saved_ranks) == len(cands['q_cap'])
        for j, group, outcome in case_indices:
            case = HuyQACaseModel(model, graph, qd, bundle, protocol, cands, ev, j,
                                  rec['alpha'], device, MAX_REL_EDGES, image_tensor=x_img)
            base = case.score_removed()
            reference = qa_scores(model, qd, x_img,
                                  np.asarray(protocol.cap[case.cap_id:case.cap_id+1]),
                                  np.asarray([case.cap_id]), case.rows_np[None], device)[0]
            np.testing.assert_allclose(base['qa'], reference, rtol=1e-4, atol=1e-5)
            query = protocol.captions[case.cap_id]['text']
            candidate_ids = tuple(str(i) for i in case.ids)
            graph_xai, scorer = case.intervention_graph, case.scorer
            before = scorer.score(query, candidate_ids, graph_xai)
            gt_id = str(int(cands['q_img'][j]))
            gt_rank = rank_of(before, gt_id, candidate_ids)
            assert gt_rank == int(saved_ranks[j]), (name, seed, case.cap_id, gt_rank, saved_ranks[j])
            for k in EXPLANATION_SIZES:
                explanation = EdgeContributionExplainer(top_k=k).explain(
                    query, str(case.top_id), candidate_ids, graph_xai, scorer)
                result = XAIEvaluator(case.schema, random_repeats=RANDOM_REPEATS,
                                      seed=SELECTION_SEED + seed*100000 + j*10 + k,
                                      match_random_edge_type=True).evaluate(
                    query, str(case.top_id), candidate_ids, graph_xai, scorer, explanation)
                after = score_without_edges(scorer, query, candidate_ids,
                                            graph_xai, explanation.edge_ids).scores
                row = {'dataset': name, 'group': group, 'seed0_outcome': outcome,
                       'caption_id': case.cap_id, 'ground_truth_image_id': gt_id,
                       'gt_rank_before': gt_rank, 'gt_rank_after': rank_of(after, gt_id, candidate_ids),
                       'requested_k': k, 'screen_coverage': len(case.screened)/max(1,len(case.local_all)),
                       'local_edges_full': len(case.local_all), 'alpha': rec['alpha'],
                       **result.to_dict()}
                all_rows.append(row)
                all_explanations.append({'dataset': name, 'seed': seed, 'caption_id': case.cap_id,
                                         'requested_k': k,
                                         'explanation': explanation_to_dict(explanation, graph_xai)})
        (OUT / f'{name}_seed{seed}.json').write_text(json.dumps(
            [r for r in all_rows if r['dataset']==name and r['seed']==seed],
            ensure_ascii=False, indent=2))
        (OUT / f'{name}_seed{seed}_explanations.json').write_text(json.dumps(
            [r for r in all_explanations if r['dataset']==name and r['seed']==seed],
            ensure_ascii=False, indent=2))
        print('Done:', name, seed, '|', sum(r['dataset']==name and r['seed']==seed for r in all_rows), 'rows')
(OUT / 'selected_cases.json').write_text(json.dumps(selected_cases, ensure_ascii=False, indent=2))
(OUT / 'explanations.json').write_text(json.dumps(all_explanations, ensure_ascii=False, indent=2))


In [ ]:
assert len(all_rows) == len(MODEL_SPECS) * len(SEEDS) * 12 * len(EXPLANATION_SIZES)
with (OUT / 'xai_results.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(all_rows[0]))
    writer.writeheader(); writer.writerows(all_rows)
summary = []
for name in MODEL_SPECS:
    for group in ('direct', 'indirect'):
        for outcome in ('success', 'failure'):
            for k in EXPLANATION_SIZES:
                subset = [r for r in all_rows if r['dataset']==name and r['group']==group
                          and r['seed0_outcome']==outcome and r['requested_k']==k]
                summary.append({'dataset': name, 'group': group, 'seed0_outcome': outcome,
                                'k': k, 'n': len(subset),
                                'mean_final_drop': statistics.fmean(r['final_score_drop'] for r in subset),
                                'mean_random_drop': statistics.fmean(r['random_final_score_drop_mean'] for r in subset),
                                'mean_advantage': statistics.fmean(r['fidelity_advantage'] for r in subset),
                                'fraction_top1_changed': statistics.fmean(r['rank_after'] > 1 for r in subset),
                                'mean_coverage': statistics.fmean(r['screen_coverage'] for r in subset)})
(OUT / 'summary.json').write_text(json.dumps({'training_manifest': str(MANIFEST),
    'n_selected_cases': len(selected_cases), 'n_rows': len(all_rows),
    'random_repeats': RANDOM_REPEATS, 'match_random_edge_type': True,
    'max_relation_edges': MAX_REL_EDGES, 'summary': summary}, ensure_ascii=False, indent=2))
for row in summary: print(row)
print('Saved:', OUT)


In [ ]:
# Chỉ dọn hai thư mục cũ của Dương sau khi cả VG và PSG đã có kết quả mới.
import shutil
assert len(all_rows) == 216 and (OUT / 'summary.json').is_file()
OLD_RUNS = [REPO / 'duong/runs/huy_qa_sage_xai',
            REPO / 'duong/runs/huy_qa_perturbation_xai']
for old in OLD_RUNS:
    assert old.resolve().is_relative_to((REPO / 'duong/runs').resolve())
    if old.is_dir():
        shutil.rmtree(old)
        print('Removed old artifact:', old)
OLD_NOTEBOOKS = ('demo_xai.ipynb', 'rebuild_huy_extra_cn_colab.ipynb',
                 'retrain_huy_qa_sage_for_xai.ipynb',
                 'huy_qa_perturbation_xai_colab.ipynb')
for name in OLD_NOTEBOOKS:
    old = REPO / 'duong/notebooks' / name
    assert old.resolve().is_relative_to((REPO / 'duong/notebooks').resolve())
    if old.is_file():
        old.unlink()
        print('Removed old notebook:', old)
OLD_MODULES = ('cli.py', 'demo_data.py', 'experiment.py', 'visualize.py')
for name in OLD_MODULES:
    old = REPO / 'duong/xai' / name
    assert old.resolve().is_relative_to((REPO / 'duong/xai').resolve())
    if old.is_file():
        old.unlink()
        print('Removed unused XAI module:', old)
old_test = REPO / 'duong/tests/test_experiment.py'
if old_test.is_file():
    old_test.unlink()
    print('Removed obsolete test:', old_test)


## Lưu ý

Kết quả mới ở `OUT`. `summary.json` là số liệu case study; CSV giữ từng caption/seed/k và `explanations.json` giữ provenance từng cạnh. Không dùng metric test để chọn giữa VG và PSG, vì hai bộ có gallery và phân bố truy vấn khác nhau. Các notebook và module demo cũ đã được thay thế trong repo; bundle gốc và kết quả của Huy không bị xóa.
